# Kilimo ADTC 2026 — Colab GPU train → GGUF → Hugging Face

LoRA fine-tunes `Qwen2.5-0.5B-Instruct` on the East African agronomy corpus, exports
GGUF Q4_K_M, smoke-tests it, and publishes it to Hugging Face.

**Run it:**

1. Runtime → Change runtime type → **T4 GPU** (free tier is fine)
2. Add a Hugging Face **write** token as a Colab Secret named `HF_TOKEN`
   (key icon in the left sidebar), then enable it for this notebook
3. Runtime → **Run all**, and expect roughly 20–40 minutes, most of it spent
   compiling `llama.cpp`

**When it finishes,** the last cell prints two lines. Paste them into
`download_model.sh` in the repo, commit, then run the profiler locally:

```bash
bash download_model.sh
adtc-profiler run --submission . --mode participant --output submission.json
python scripts/score.py submission.json
```

Uploads to `rssebambulidde/adtc-kilimo-0.5b-gguf` as `adtc-kilimo-0.5b-q4_k_m.gguf`.
The repo must be **public** — the evaluator fetches the weights without credentials,
and the last cell verifies that anonymous fetch works.

T4 has no bfloat16, so `train_lora.py` automatically downgrades to fp16.

In [ ]:
# Paste a Hugging Face write token, or use Colab Secrets named HF_TOKEN
import os
from google.colab import userdata

HF_TOKEN = os.environ.get("HF_TOKEN") or ""
try:
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass

assert HF_TOKEN, "Set HF_TOKEN (HF write token) before running"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

HF_USER = "rssebambulidde"
HF_REPO = f"{HF_USER}/adtc-kilimo-0.5b-gguf"
GGUF_NAME = "adtc-kilimo-0.5b-q4_k_m.gguf"
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
print("HF repo target:", HF_REPO)

In [ ]:
!nvidia-smi
%pip install -q "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.19" accelerate safetensors pyyaml huggingface_hub hf_transfer sentencepiece protobuf

# Report the compute capability up front. Free Colab usually allocates a T4, which is
# Turing and has no bfloat16; train_lora.py downgrades to fp16 automatically.
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported(), "-> training will use",
          "bf16" if torch.cuda.is_bf16_supported() else "fp16")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

In [ ]:
# Clone submission repo (public or your fork). Override if private.
REPO_URL = "https://github.com/rssebambulidde/adtc-2026.git"
import os, pathlib
if not pathlib.Path("adtc-2026").exists():
    !git clone --depth 1 {REPO_URL} adtc-2026
%cd adtc-2026
!python scripts/build_dataset.py --out data/build/train.jsonl --augment
!wc -l data/build/train.jsonl

In [ ]:
!python scripts/train_lora.py --config configs/kilimo-0.5b.yaml

In [ ]:
# Merge LoRA → full HF weights
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter = "checkpoints/kilimo-0.5b-lora"
merged = Path("merged/kilimo-0.5b")
merged.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(adapter, trust_remote_code=True)
# float32 for the merge: a 0.5B model is only ~2 GB in fp32, so there is no reason to
# add a lossy half-precision hop ahead of the f16 GGUF conversion.
model = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float32, device_map="cpu", trust_remote_code=True
)
model = PeftModel.from_pretrained(model, adapter)
model = model.merge_and_unload()
model.save_pretrained(merged, safe_serialization=True)
tokenizer.save_pretrained(merged)
print("merged ->", merged)

In [ ]:
# Convert + quantize with llama.cpp
%cd /content
import pathlib
if not pathlib.Path("llama.cpp").exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp
%cd llama.cpp

# llama-cli is built too so the quantized model can be smoke-tested before upload.
!cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF
!cmake --build build --config Release -j --target llama-quantize llama-cli
!pip install -q -r requirements/requirements-convert_hf_to_gguf.txt

!mkdir -p /content/adtc-2026/model
!python convert_hf_to_gguf.py /content/adtc-2026/merged/kilimo-0.5b \
  --outfile /content/adtc-2026/model/adtc-kilimo-0.5b-f16.gguf --outtype f16

!./build/bin/llama-quantize \
  /content/adtc-2026/model/adtc-kilimo-0.5b-f16.gguf \
  /content/adtc-2026/model/adtc-kilimo-0.5b-q4_k_m.gguf Q4_K_M

!ls -lh /content/adtc-2026/model/*.gguf

In [ ]:
# Smoke test the quantized weights on the real submission prompt BEFORE uploading.
# A bad merge or a broken chat template produces fluent nonsense, an empty reply, or
# endless repetition, and all of those are cheap to catch here and expensive later.
import json, subprocess

meta = json.load(open("/content/adtc-2026/metadata.json"))
prompt = meta["test_prompts"][0]["prompt"]
gguf = "/content/adtc-2026/model/adtc-kilimo-0.5b-q4_k_m.gguf"

out = subprocess.run(
    ["./build/bin/llama-cli", "-m", gguf, "-p", prompt,
     "-n", "220", "-ngl", "0", "--temp", "0.7", "--no-warmup", "-no-cnv"],
    capture_output=True, text=True, timeout=900,
).stdout

reply = out[len(prompt):] if prompt in out else out
print(reply.strip()[:1500])

words = reply.split()
print("\n" + "=" * 60)
print(f"tokens-ish: {len(words)}")
if len(words) < 25:
    print("FAIL: reply too short — check the merge and the chat template.")
elif len(set(words)) < len(words) / 4:
    print("FAIL: heavy repetition — likely a broken merge or bad LoRA.")
else:
    print("PASS: fluent, non-degenerate reply. Safe to upload.")
print("Read it yourself — fluency is not correctness. Agronomy must be right.")

In [ ]:
import hashlib, os, pathlib
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import HfApi, create_repo

path = pathlib.Path(f"/content/adtc-2026/model/{GGUF_NAME}")
assert path.is_file(), f"missing {path} — run the quantize cell first"

# Hash before upload: this is the value the evaluator's download_model.sh verifies,
# so it must describe the exact bytes that were sent.
sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
print("local sha256:", sha256)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=str(path),
    path_in_repo=GGUF_NAME,
    repo_id=HF_REPO,
    token=HF_TOKEN,
)
readme = f"""---
license: apache-2.0
base_model: {BASE}
tags:
- gguf
- agriculture
- adtc-2026
---
# ADTC Kilimo 0.5B (Q4_K_M)

LoRA-tuned Qwen2.5-0.5B-Instruct for East African smallholder agriculture advisory.\n
File: `{GGUF_NAME}`\n
"""
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo="README.md",
    repo_id=HF_REPO,
    token=HF_TOKEN,
)

# Confirm the upload is publicly fetchable without credentials, which is exactly how
# the evaluator will fetch it. A private repo silently breaks the whole submission.
url = f"https://huggingface.co/{HF_REPO}/resolve/main/{GGUF_NAME}"
import urllib.request
try:
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=60) as r:
        print(f"public fetch OK: HTTP {r.status}, {int(r.headers.get('content-length', 0)) / 1e6:.1f} MB")
except Exception as e:
    print(f"PUBLIC FETCH FAILED: {e}\n-> make the repo public on huggingface.co before submitting")

print("\n" + "=" * 66)
print("Paste these two lines into download_model.sh:\n")
print(f'MODEL_REPO="${{MODEL_REPO:-{HF_REPO}}}"')
print(f'EXPECTED_SHA256="${{EXPECTED_SHA256:-{sha256}}}"')
print("=" * 66)